# Analysis of EACL 2027 Agent Runs
This notebook analyzes the `outputs/runs.jsonl` file. It evaluates the performance, diversity, and cost of different search fan-out strategies.

In [ ]:
# Install necessary visualization libraries, bypassing Corp Airlock if necessary
!pip install --index-url https://pypi.org/simple pandas matplotlib seaborn

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Load the JSONL file
data = []
try:
    with open('outputs/runs.jsonl', 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    print(f"Loaded {len(data)} runs.")
except FileNotFoundError:
    print("outputs/runs.jsonl not found. Make sure you have executed run_agent.py at least once.")

df = pd.DataFrame(data)
if not df.empty:
    display(df.head())

## 1. Information Overlap (Diversity)
Are the personalized and disconfirming branches actually finding *new* information, or are they just returning the same URLs as the generic searches?

In [ ]:
if not df.empty and 'raw_search_results' in df.columns:
    # Flatten the search results to analyze duplicates
    results_list = []
    for index, row in df.iterrows():
        run_id = row['run_id']
        variant = row['variant']
        for res in row.get('raw_search_results', []):
            results_list.append({
                'run_id': run_id,
                'variant': variant,
                'branch_type': res.get('branch_type'),
                'is_duplicate': res.get('is_duplicate_url', False),
                'score': res.get('score')
            })
            
    results_df = pd.DataFrame(results_list)
    
    # Calculate percentage of duplicate URLs per branch type
    if not results_df.empty:
        dup_stats = results_df.groupby('branch_type')['is_duplicate'].mean().reset_index()
        dup_stats['is_duplicate'] = dup_stats['is_duplicate'] * 100
        
        plt.figure(figsize=(8, 5))
        ax = sns.barplot(data=dup_stats, x='branch_type', y='is_duplicate', palette='viridis')
        plt.title('Percentage of Duplicate URLs by Branch Type')
        plt.ylabel('% of Search Results that were Duplicates')
        plt.xlabel('Branch Type')
        plt.xticks(rotation=45)
        
        # Add labels on top of bars
        for p in ax.patches:
            ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', fontsize=10, color='black', xytext=(0, 5),
                        textcoords='offset points')
        plt.tight_layout()
        plt.show()

## 2. Cost Analysis
How many LLM calls and Search calls does each variant make?

In [ ]:
if not df.empty and 'cost_proxy' in df.columns:
    # Extract cost metrics
    cost_data = []
    for index, row in df.iterrows():
        cost = row.get('cost_proxy', {})
        if cost:
            cost_data.append({
                'run_id': row['run_id'],
                'variant': row['variant'],
                'llm_calls': cost.get('num_gemini_calls', 0),
                'search_calls': cost.get('num_tavily_calls', 0),
                'total_results': cost.get('num_raw_results', 0)
            })
            
    cost_df = pd.DataFrame(cost_data)
    
    if not cost_df.empty:
        # Plot LLM and Search calls side-by-side
        cost_melted = pd.melt(cost_df, id_vars=['variant'], value_vars=['llm_calls', 'search_calls'], 
                              var_name='Call Type', value_name='Number of Calls')
                              
        plt.figure(figsize=(10, 6))
        sns.barplot(data=cost_melted, x='variant', y='Number of Calls', hue='Call Type', errorbar=None, palette='mako')
        plt.title('Cost Metrics (API Calls) per Variant')
        plt.xlabel('Experiment Variant')
        plt.ylabel('Number of API Calls')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()